# Testing the nonlinear capabilities of `PDESolver`

This notebook exercises the `_apply_nonlinear` method by solving a collection of non-trivial nonlinear PDEs **without** requiring exact solutions. The goal is to verify that all derivative banks are correctly handled:

- **1D**: `u_x`, `u_xx`, `u_xxx`
- **2D**: `u_x`, `u_y`, `u_xx`, `u_yy`, `u_xy`

**Design principles for stable nonlinear tests**

- Use **periodic initial conditions** on periodic domains to avoid Gibbs pollution from discontinuous boundary transitions.
- Keep **amplitudes small** for high-order derivative nonlinearities — stiffness grows as $|\hat{u}| \cdot k^n$.
- Use `time_scheme='ETD-RK4'` for better stability — the exponential integrator handles the stiff linear part exactly.
- Increase `Nt` for aggressive nonlinearities and reduce `Lt` to stay within the solution's validity window.
- Use `component='real'` for real-valued fields — `'abs'` is only meaningful for complex-valued solutions.

**Prerequisites**  
The files `solver.py`, `psiop.py`, and `imports.py` must be in the same directory (or on the Python path).

In [ ]:
from solver import *
%matplotlib inline

## 1D test cases

Common settings: $L_x = 2\pi$, $N_x = 256$, `ETD-RK4` time scheme.

All initial conditions are **fully periodic** (trigonometric) to avoid spectral pollution at the domain boundaries.

In [ ]:
x, t = symbols('x t')
u = Function('u')

lx = 2 * np.pi
nx = 256

def run_1d(eq, ic, title='', lt=5, nt=600, n_frames=40):
    """Solve a 1D PDE and return an animation.
    
    Uses ETD-RK4 throughout for better nonlinear stability.
    Nt and Lt can be tuned per equation.
    """
    solver = PDESolver(eq, time_scheme='ETD-RK4')
    solver.setup(
        Lx=lx, Nx=nx, Lt=lt, Nt=nt,
        initial_condition=ic,
        n_frames=n_frames,
        plot=False
    )
    solver.solve()
    print(f'Solved: {title}')
    return solver.animate(mode='plot', component='real')

### 1) Triple-derivative nonlinearity

$$\partial_t u + u\,\partial_{xxx} u = 0$$

This tests the `u_xxx` bank. The equation is dispersive with a nonlinear coupling: energy cascades across wavenumbers. A **multi-modal periodic IC** avoids the non-periodic Gaussian artifact and provides a richer initial spectral content to reveal dispersive behaviour.

**Stability note**: `u·u_xxx` grows as $\hat{u} \cdot k^3$ — very stiff at high wavenumbers. Small amplitude and moderate `Lt` are critical.

In [ ]:
eq1 = Eq(Derivative(u(t,x), t) + u(t,x) * Derivative(u(t,x), (x,3)) - 0.01* Derivative(u(t,x), (x, 2)), 0)

# Multi-modal periodic IC: small amplitude keeps u·u_xxx manageable
ic1 = lambda x: 0.3 * np.sin(x) + 0.15 * np.cos(2*x) + 0.1 * np.sin(3*x)

ani1 = run_1d(eq1, ic1, 'u_t + u u_xxx = 0', lt=3.3, nt=800)
HTML(ani1.to_jshtml())

### 2) Gradient-gradient product

$$\partial_t u = \partial_x u\;\partial_{xx} u$$

Note that $u_x\,u_{xx} = \frac{1}{2}\partial_x(u_x^2)$, so this is a **conservative** nonlinearity that redistributes energy in $u_x$. It can form shocks in finite time, so a moderate amplitude and enough time steps are essential.

A periodic IC with a dominant low wavenumber gives a clean shock-formation scenario.

In [ ]:
eq2 = Eq(
    Derivative(u(t,x), t),
    Derivative(u(t,x), x) * Derivative(u(t,x), (x,2)) + 0.01* Derivative(u(t,x), (x, 2))
)

# Asymmetric periodic IC to trigger non-trivial shock formation
ic2 = lambda x: 0.5 * np.sin(x) + 0.2 * np.sin(2*x) + 0.1 * np.cos(3*x)

ani2 = run_1d(eq2, ic2, 'u_t = u_x u_xx', lt=1, nt=800)
HTML(ani2.to_jshtml())

### 3) Functional nonlinearity — nonlinear diffusion

$$\partial_t u = \sin(u)\,\partial_{xx} u$$

This is a **nonlinear diffusion** equation: the diffusivity is $\sin(u)$, which is positive for $u \in (0, \pi)$, zero at $u=0$ and $u=\pi$, and negative outside — meaning diffusion can change sign and create instabilities if the amplitude leaves $(0,\pi)$.

The IC is chosen to stay comfortably in $(0, \pi/2)$ where $\sin(u) > 0$, guaranteeing well-posed diffusive behaviour. A **DC offset** (mean value $> 0$) ensures $\sin(u)$ never crosses zero.

In [ ]:
eq3 = Eq(
    Derivative(u(t,x), t),
    sin(u(t,x)) * Derivative(u(t,x), (x,2))
    + 0.01* Derivative(u(t,x), (x, 2))
)

# DC offset keeps u in (0, pi/2): sin(u) > 0 throughout → pure diffusion
ic3 = lambda x: np.pi/3 + 0.3 * np.sin(x) + 0.15 * np.cos(2*x)

ani3 = run_1d(eq3, ic3, 'u_t = sin(u) u_xx', lt=3.0, nt=600)
HTML(ani3.to_jshtml())

### 4) Composite nonlinearity — two additive terms

$$\partial_t u = u^2\,\partial_{xxx} u + u\,\partial_x u\,\partial_{xx} u$$

This tests **multiple nonlinear terms** being accumulated simultaneously. The first term is dispersive (third derivative), the second is shock-forming (gradient product). Their interaction is non-trivial.

**Critical**: the $u^2\,u_{xxx}$ term is extremely stiff — amplitude must be kept very small ($\lesssim 0.2$) and $\Delta t$ must be tiny. Use high `Nt` and short `Lt`.

In [ ]:
eq4 = Eq(
    Derivative(u(t,x), t),
    u(t,x)**2 * Derivative(u(t,x), (x,3))
    + u(t,x) * Derivative(u(t,x), x) * Derivative(u(t,x), (x,2))
    + 0.01 * Derivative(u(t,x), (x, 2))
)

# Small amplitude: u^2 u_xxx grows as amplitude^2 * k^3 — very aggressive
ic4 = lambda x: 0.15 * np.sin(x) + 0.08 * np.cos(2*x)

ani4 = run_1d(eq4, ic4, 'u_t = u^2 u_xxx + u u_x u_xx', lt=5.2, nt=1000)
HTML(ani4.to_jshtml())

## 2D test cases

Common settings: $L_x = L_y = 2\pi$, $N_x = N_y = 128$, `ETD-RK4`.

All 2D ICs are **doubly periodic** (products/sums of trigonometric functions). Gaussians are explicitly avoided because $e^{-x^2-y^2}$ is not periodic and causes spectral leakage across the entire wavenumber space, polluting the high-$k$ modes and triggering spurious blow-up.

Animations use `mode='imshow'` with `overlay='contour'` and `component='real'`.

In [ ]:
x, y, t = symbols('x y t')
u = Function('u')

lx, ly = 2*np.pi, 2*np.pi
nx, ny = 128, 128

def run_2d(eq, ic, title='', lt=0.5, nt=400, n_frames=40):
    """Solve a 2D PDE and return an imshow animation with contour overlay."""
    solver = PDESolver(eq, time_scheme='ETD-RK4')
    solver.setup(
        Lx=lx, Ly=ly, Nx=nx, Ny=ny,
        Lt=lt, Nt=nt,
        initial_condition=ic,
        n_frames=n_frames,
        plot=False
    )
    solver.solve()
    print(f'Solved 2D: {title}')
    return solver.animate(mode='imshow', overlay='contour', component='real')

### 5) Cross-derivative advection

$$\partial_t u + u\,\partial_{xy} u = 0$$

Tests the mixed derivative `u_xy` bank. The $u\,u_{xy}$ term couples x- and y-wavenumbers nonlinearly, producing an anisotropic energy transfer. A doubly periodic IC with anisotropic structure reveals this coupling clearly.

In [ ]:
eq5 = Eq(
    Derivative(u(t,x,y), t)
    + u(t,x,y) * Derivative(u(t,x,y), x, y) + 0.01* (Derivative(u(t,x,y), (x, 2)) + Derivative(u(t,x,y), (y, 2))),
    0
)

# Doubly periodic, anisotropic: emphasises x-y coupling
ic5 = lambda x, y: (
    0.4 * np.sin(x) * np.cos(y)
    + 0.2 * np.cos(2*x) * np.sin(y)
    + 0.1 * np.sin(x) * np.sin(2*y)
)

ani5 = run_2d(eq5, ic5, 'u_t + u u_xy = 0', lt=0.4, nt=600)
HTML(ani5.to_jshtml())

### 6) Full 2D derivative product

$$\partial_t u = \partial_x u\;\partial_y u\;\partial_{xx} u\;\partial_{yy} u\;\partial_{xy} u$$

Tests **all five 2D derivative banks simultaneously**. This is an extremely aggressive nonlinearity — a product of five derivative fields — and will be negligibly small only when $u$ is small. A very small amplitude, short `Lt`, and high `Nt` are mandatory.

The IC uses several modes to populate all relevant wavenumbers, but with amplitude $\ll 1$.

In [ ]:
eq6 = Eq(
    Derivative(u(t,x,y), t),
    Derivative(u(t,x,y), x)
    * Derivative(u(t,x,y), y)
    * Derivative(u(t,x,y), (x,2))
    * Derivative(u(t,x,y), (y,2))
    * Derivative(u(t,x,y), x, y)
    + 0.01* (Derivative(u(t,x,y), (x, 2)) + Derivative(u(t,x,y), (y, 2)))
)

# Very small amplitude: product of 5 derivatives makes this extremely stiff
# Periodic IC with modest wavenumber content
ic6 = lambda x, y: (
    0.1 * np.sin(x) * np.cos(y)
    + 0.05 * np.cos(2*x) * np.sin(2*y)
    + 0.03 * np.sin(x + y)
)

ani6 = run_2d(eq6, ic6, 'u_t = u_x u_y u_xx u_yy u_xy', lt=10.1, nt=800)
HTML(ani6.to_jshtml())

### 7) Two-term parallel nonlinearity

$$\partial_t u = u^2\,\partial_{xx} u + \partial_x u\,\partial_y u$$

Tests the **ThreadPoolExecutor path** (two independent nonlinear terms evaluated in parallel). The first term $u^2\,u_{xx}$ is a nonlinear diffusion/anti-diffusion, the second $u_x\,u_y$ is an anisotropic advective coupling.

A doubly periodic IC with both x and y structure ensures both terms contribute meaningfully from the start.

In [ ]:
eq7 = Eq(
    Derivative(u(t,x,y), t),
    u(t,x,y)**2 * Derivative(u(t,x,y), (x,2))
    + Derivative(u(t,x,y), x) * Derivative(u(t,x,y), y)
    + 0.01* (Derivative(u(t,x,y), (x, 2)) + Derivative(u(t,x,y), (y, 2)))
)

# Doubly periodic with both x and y content; moderate amplitude for u^2 u_xx
ic7 = lambda x, y: (
    0.3 * np.sin(x) * np.sin(y)
    + 0.15 * np.cos(2*x) * np.cos(y)
    + 0.1 * np.sin(x) * np.cos(3*y)
)

ani7 = run_2d(eq7, ic7, 'u_t = u^2 u_xx + u_x u_y', lt=10.5, nt=600)
HTML(ani7.to_jshtml())

In [ ]:
eq7 = Eq(
    Derivative(u(t,x,y), t),
    Derivative(u(t,x,y), x) * Derivative(u(t,x,y), y) + 0.01* (Derivative(u(t,x,y), (x, 2)) + Derivative(u(t,x,y), (y, 2)))
)

# Doubly periodic with both x and y content; moderate amplitude for u^2 u_xx
ic7 = lambda x, y: (
    0.3 * np.sin(x) * np.sin(y)
    + 0.15 * np.cos(2*x) * np.cos(y)
    + 0.1 * np.sin(x) * np.cos(3*y)
)

ani7 = run_2d(eq7, ic7, 'u_t = u^2 u_xx + u_x u_y', lt=5.5, nt=600)
HTML(ani7.to_jshtml())